# MLP

Neste exercício estudei o que uma camada oculta acrescenta à rede. Primeiro fiz as contas de uma atualização de backpropagation e depois usei uma MLP em NumPy para classificação binária, multiclasse e comparação de profundidade.

Fixei os geradores dos dados e os splits em random_state=42. As inicializações usam um único rng criado com `np.random.default_rng(42)`. Na comparação pareada de profundidade, reinicio o estado desse mesmo rng para cada modelo e seed. As seeds adicionais são 43 e 44, elas mudam somente os pesos iniciais. Mantive os dados, o split, a padronização, a taxa de aprendizado e o número de épocas fixos.

In [1]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs, make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

# Funciona como script ou notebook, na raiz do repo ou na pasta da entrega.
if "__file__" in globals():
    BASE = Path(__file__).resolve().parents[1]
else:
    BASE = Path.cwd()
    if not (BASE / "code" / "mlp.py").exists():
        BASE = BASE / "docs" / "exercises" / "mlp"
sys.path.insert(0, str(BASE / "code"))
from mlp import MLP

FIGURES = BASE / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(42)
COLORS = ["#2471a3", "#d35400", "#239b56"]
LEARNING_RATE = 0.1
EPOCHS = 2000
results = {}
plt.rcParams.update({"font.size": 11, "figure.dpi": 120})


def save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(FIGURES / filename, dpi=160)
    plt.close(fig)


def standardize(X_train, X_test):
    # Média e desvio são calculados só no treino: o teste não influencia o ajuste.
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    return (X_train - mean) / std, (X_test - mean) / std, mean, std


def accuracy(model, X, y):
    return float(np.mean(model.predict(X) == y))


def loss_plot(history, title, label, filename):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(np.arange(len(history)), history, label=label)
    ax.set(title=title, xlabel="Época (0 = antes de treinar)", ylabel="Entropia cruzada média")
    ax.legend()
    save_figure(fig, filename)

## Exercise 1

Uma etapa de backpropagation, com as contas abertas.

### A — Forward pass

Aqui os pesos da camada oculta têm um neurônio por linha, como no enunciado. A entrada é x = [0,5; −0,2], o alvo é y = 1 e a ativação das duas camadas é tanh. Mantive mais casas durante as contas e arredondei os resultados apresentados para seis casas.

A pré-ativação do primeiro neurônio é:

$$
z^{(1)}_1 = 0{,}3(0{,}5) + (-0{,}1)(-0{,}2) + 0{,}1
= 0{,}150000 + 0{,}020000 + 0{,}100000 = 0{,}270000.
$$

Para o segundo neurônio:

$$
z^{(1)}_2 = 0{,}2(0{,}5) + 0{,}4(-0{,}2) - 0{,}2
= 0{,}100000 - 0{,}080000 - 0{,}200000 = -0{,}180000.
$$

Aplicando tanh, as ativações são:

$$
h^{(1)}_1 = \tanh(0{,}270000) = 0{,}263625,
\qquad h^{(1)}_2 = \tanh(-0{,}180000) = -0{,}178081.
$$

A saída antes da ativação é:

$$
u^{(2)} = 0{,}5(0{,}263625) + (-0{,}3)(-0{,}178081) + 0{,}2
= 0{,}131812 + 0{,}053424 + 0{,}200000 = 0{,}385237.
$$

Então a previsão e a perda são:

$$
\hat y = \tanh(0{,}385237) = 0{,}367247,
\qquad
L = (1 - 0{,}367247)^2 = (0{,}632753)^2 = 0{,}400377.
$$

Assim, z¹ = [0,270000; −0,180000], h¹ = [0,263625; −0,178081], u² = 0,385237, ŷ = 0,367247 e L = 0,400377.

### B — Backward pass

Começo pela derivada da perda em relação à previsão:

$$
\frac{\partial L}{\partial \hat y} = 2(\hat y-y)
= 2(0{,}367247-1) = -1{,}265507.
$$

Como a derivada de tanh é 1 menos sua saída ao quadrado, na camada de saída:

$$
\frac{\partial \hat y}{\partial u^{(2)}} = 1-\hat y^2 = 0{,}865130.
$$

Pela regra da cadeia, o gradiente da pré-ativação é:

$$
\delta^{(2)} = \frac{\partial L}{\partial u^{(2)}}
= \frac{\partial L}{\partial \hat y}\frac{\partial \hat y}{\partial u^{(2)}}
= (-1{,}265507)(0{,}865130) = -1{,}094828.
$$

Cada peso da saída multiplica uma ativação oculta. Por isso:

$$
\frac{\partial L}{\partial W^{(2)}_1}
= \delta^{(2)}h^{(1)}_1
= (-1{,}094828)(0{,}263625) = -0{,}288624,
$$

$$
\frac{\partial L}{\partial W^{(2)}_2}
= \delta^{(2)}h^{(1)}_2
= (-1{,}094828)(-0{,}178081) = 0{,}194968.
$$

O bias entra somando na pré-ativação e sua derivada local é 1:

$$
\frac{\partial L}{\partial b^{(2)}} = \delta^{(2)} = -1{,}094828.
$$

Voltando para as ativações ocultas, uso os pesos anteriores à atualização:

$$
\frac{\partial L}{\partial h^{(1)}_1}
= \delta^{(2)}W^{(2)}_1 = (-1{,}094828)(0{,}5) = -0{,}547414,
$$

$$
\frac{\partial L}{\partial h^{(1)}_2}
= \delta^{(2)}W^{(2)}_2 = (-1{,}094828)(-0{,}3) = 0{,}328448.
$$

As derivadas locais dos dois neurônios são 1 − h₁² = 0,930502 e 1 − h₂² = 0,968287. Então:

$$
\delta^{(1)}_1 = \frac{\partial L}{\partial z^{(1)}_1}
= (-0{,}547414)(0{,}930502) = -0{,}509370,
$$

$$
\delta^{(1)}_2 = \frac{\partial L}{\partial z^{(1)}_2}
= (0{,}328448)(0{,}968287) = 0{,}318032.
$$

Para cada peso oculto, a derivada local é a entrada que ele multiplica: ∂L/∂W¹ᵢⱼ = δ¹ᵢxⱼ. Abrindo os quatro valores:

$$
\frac{\partial L}{\partial W^{(1)}_{11}} = (-0{,}509370)(0{,}5) = -0{,}254685,
\qquad
\frac{\partial L}{\partial W^{(1)}_{12}} = (-0{,}509370)(-0{,}2) = 0{,}101874,
$$

$$
\frac{\partial L}{\partial W^{(1)}_{21}} = (0{,}318032)(0{,}5) = 0{,}159016,
\qquad
\frac{\partial L}{\partial W^{(1)}_{22}} = (0{,}318032)(-0{,}2) = -0{,}063606.
$$

Nos biases ocultos a derivada local também é 1, então ∂L/∂b¹₁ = −0,509370 e ∂L/∂b¹₂ = 0,318032. Os gradientes completos ficam:

$$
\frac{\partial L}{\partial W^{(1)}} =
\begin{bmatrix}-0{,}254685 & 0{,}101874\\0{,}159016 & -0{,}063606\end{bmatrix},
\quad
\frac{\partial L}{\partial b^{(1)}} = [-0{,}509370;\ 0{,}318032],
$$

$$
\frac{\partial L}{\partial W^{(2)}} = [-0{,}288624;\ 0{,}194968],
\quad
\frac{\partial L}{\partial b^{(2)}} = -1{,}094828.
$$

### C — Parameter update

A atualização é θ novo = θ antigo − 0,3 ∂L/∂θ. Na saída:

$$
W^{(2)}_1 = 0{,}5 - 0{,}3(-0{,}288624) = 0{,}586587,
\qquad
W^{(2)}_2 = -0{,}3 - 0{,}3(0{,}194968) = -0{,}358490,
$$

$$
b^{(2)} = 0{,}2 - 0{,}3(-1{,}094828) = 0{,}528448.
$$

Na camada oculta:

$$
W^{(1)}_{11} = 0{,}3 - 0{,}3(-0{,}254685) = 0{,}376405,
\qquad
W^{(1)}_{12} = -0{,}1 - 0{,}3(0{,}101874) = -0{,}130562,
$$

$$
W^{(1)}_{21} = 0{,}2 - 0{,}3(0{,}159016) = 0{,}152295,
\qquad
W^{(1)}_{22} = 0{,}4 - 0{,}3(-0{,}063606) = 0{,}419082,
$$

$$
b^{(1)}_1 = 0{,}1 - 0{,}3(-0{,}509370) = 0{,}252811,
\qquad
b^{(1)}_2 = -0{,}2 - 0{,}3(0{,}318032) = -0{,}295410.
$$

São nove parâmetros escalares: quatro pesos ocultos, dois biases ocultos, dois pesos de saída e um bias de saída. O enunciado menciona oito, mas atualizei todos os valores fornecidos.

Em um segundo forward pass, as ativações ocultas ficam [0,435874; −0,294127], a pré-ativação da saída fica 0,889568 e a previsão sobe para 0,711181. A perda cai de 0,400377 para 0,083417. Neste passo, mover os parâmetros contra o gradiente aproximou a previsão do alvo 1. Isso confirma a redução para a taxa escolhida; não significa que qualquer taxa de aprendizado faria a perda cair.

## Exercise 2

Classificação binária com uma classe dividida em dois grupos.

### A — Generate the data

Gerei exatamente os 1.000 pontos pedidos. A classe 0 tem 500 pontos ao redor de [0; 0]. A classe 1 junta 250 pontos ao redor de [3; 3] e 250 ao redor de [−3; −3], com desvio 1,2 em cada grupo. A figura 1 mostra a classe 0 no meio das duas partes da classe 1.

Separei 800 pontos para treino e 200 para teste, com estratificação. O treino tem 400 pontos de cada classe e o teste tem 100 de cada. Padronizei as entradas com a média e o desvio do treino, aplicando os mesmos valores ao teste. Isso ajuda a tanh a trabalhar em uma escala adequada, sem mudar a geometria que impede a separação linear.

In [2]:
centers = np.array([[0., 0.], [3., 3.], [-3., -3.]])
X_binary, c = make_blobs(n_samples=[500, 250, 250], centers=centers,
                         cluster_std=1.2, random_state=42)
y_binary = (c > 0).astype(int)
Xb_train_raw, Xb_test_raw, yb_train, yb_test = train_test_split(
    X_binary, y_binary, test_size=0.2, random_state=42, stratify=y_binary)
Xb_train, Xb_test, mean_binary, std_binary = standardize(Xb_train_raw, Xb_test_raw)

fig, ax = plt.subplots(figsize=(7, 5))
for label in range(2):
    points = X_binary[y_binary == label]
    ax.scatter(points[:, 0], points[:, 1], s=15, alpha=0.6,
               c=COLORS[label], label=f"Classe {label}")
ax.set(title="Figura 1 — Dados de classificação binária", xlabel="x1", ylabel="x2")
ax.legend()
save_figure(fig, "fig01-dados.png")
print("Binário: treino", Xb_train.shape, "; teste", Xb_test.shape)
print("Classes no treino/teste:", np.bincount(yb_train), np.bincount(yb_test))

Binário: treino (800, 2) ; teste (200, 2)
Classes no treino/teste: [400 400] [100 100]


![Figura 1 — Dados binários, com os 1.000 pontos separados por classe.](figures/fig01-dados.png)

Figura 1 — A classe 0 ocupa o centro e a classe 1 aparece em dois grupos opostos.

### B — Establish the baseline

Usei regressão logística como baseline linear, com as mesmas entradas padronizadas e o mesmo split. Ela acertou 95 dos 200 pontos de teste, ou 47,50%. Como o teste está balanceado, responder sempre uma classe daria 50,00%.

In [3]:
baseline_binary = LogisticRegression(random_state=42).fit(Xb_train, yb_train)
results["baseline_binary"] = accuracy(baseline_binary, Xb_test, yb_test)
print(f"Baseline linear binário: {100 * results['baseline_binary']:.2f}%")

Baseline linear binário: 47.50%


### C — Implement the MLP

Escolhi a arquitetura 2–16–1: duas entradas, 16 neurônios ocultos com tanh e uma saída sigmoid. São 65 parâmetros, calculados por 2×16 + 16 + 16×1 + 1. Treinei por 2.000 épocas, com taxa 0,1 e descida do gradiente usando todo o treino em cada atualização. Inicializei os pesos com Xavier e os biases com zero. Não usei regularização, momentum nem parada antecipada.

Para cada camada oculta, calculo Z = AW + b e H = tanh(Z). No código, cada amostra ocupa uma linha e cada neurônio uma coluna dos pesos. É a transposta da convenção usada no exercício 1, mas faz a mesma conta.

Na saída binária, p = sigmoid(u). Para uma amostra, a entropia cruzada é:

$$
\ell = -[y\log p + (1-y)\log(1-p)].
$$

Sua derivada em relação a p é −y/p + (1−y)/(1−p). Multiplicando pela derivada da sigmoid, p(1−p), obtenho:

$$
\frac{\partial\ell}{\partial u}
= \left(-\frac{y}{p}+\frac{1-y}{1-p}\right)p(1-p)
= -y(1-p)+(1-y)p = p-y.
$$

Como a perda usada é a média de N amostras, o delta da saída é (P−Y)/N. Para qualquer camada, os gradientes e a propagação para a camada anterior são:

$$
\frac{\partial L}{\partial W^{(l)}} = (A^{(l-1)})^T\Delta^{(l)},
\qquad
\frac{\partial L}{\partial b^{(l)}} = \sum_{i=1}^{N}\Delta^{(l)}_i,
$$

$$
\Delta^{(l-1)} = [\Delta^{(l)}(W^{(l)})^T]\odot[1-(A^{(l-1)})^2].
$$

A multiplicação por A vem da derivada da combinação linear em relação aos pesos. O bias recebe a soma porque é compartilhado entre as amostras. A última expressão aplica a regra da cadeia e a derivada da tanh. Calculo todos esses gradientes antes de alterar qualquer peso e depois faço W ← W − η∂L/∂W e b ← b − η∂L/∂b.

A classe abaixo recebe os tamanhos das camadas como argumento. Ela já trata saídas binárias e multiclasse, então nos próximos exercícios só preciso mudar os tamanhos. O forward, o backward e o update continuam exatamente iguais.

In [4]:
"""MLP em NumPy: o mesmo forward, backward e update para qualquer profundidade."""

import numpy as np


class MLP:
    def __init__(self, layer_sizes, rng, learning_rate=0.1, epochs=2000):
        self.layer_sizes = list(layer_sizes)
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.weights = []
        self.biases = []
        for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
            # Xavier mantém a escala dos sinais adequada para a tanh.
            limit = np.sqrt(6 / (n_in + n_out))
            self.weights.append(rng.uniform(-limit, limit, (n_in, n_out)))
            self.biases.append(np.zeros((1, n_out)))
        self.loss_history = []

    def forward(self, X):
        # Cada linha é uma amostra; por isso usamos X @ W, e não W @ x.
        activations = [X]
        for W, b in zip(self.weights[:-1], self.biases[:-1]):
            activations.append(np.tanh(activations[-1] @ W + b))
        logits = activations[-1] @ self.weights[-1] + self.biases[-1]
        if self.layer_sizes[-1] == 1:
            # Forma estável da sigmoid, sem overflow de exp(-z).
            probabilities = (1 + np.tanh(logits / 2)) / 2
        else:
            shifted = logits - logits.max(axis=1, keepdims=True)
            exp_logits = np.exp(shifted)
            probabilities = exp_logits / exp_logits.sum(axis=1, keepdims=True)
        activations.append(probabilities)
        return logits, activations

    def loss(self, logits, y):
        if self.layer_sizes[-1] == 1:
            # BCE = log(1 + exp(u)) - y*u, calculada de forma estável.
            return float(np.mean(np.logaddexp(0, logits) - y.reshape(-1, 1) * logits))
        shifted = logits - logits.max(axis=1, keepdims=True)
        log_probabilities = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
        return float(-np.mean(log_probabilities[np.arange(len(y)), y]))

    def backward(self, activations, y):
        probabilities = activations[-1]
        if self.layer_sizes[-1] == 1:
            target = y.reshape(-1, 1)
        else:
            target = np.eye(self.layer_sizes[-1])[y]
        # Sigmoid+BCE e softmax+CE têm o mesmo gradiente dos logits.
        delta = (probabilities - target) / len(y)
        grad_weights = [None] * len(self.weights)
        grad_biases = [None] * len(self.biases)
        for layer in range(len(self.weights) - 1, -1, -1):
            grad_weights[layer] = activations[layer].T @ delta
            grad_biases[layer] = delta.sum(axis=0, keepdims=True)
            if layer > 0:
                # Ainda usamos os pesos antigos: o update só ocorre depois.
                delta = (delta @ self.weights[layer].T) * (1 - activations[layer] ** 2)
        return grad_weights, grad_biases

    def update(self, grad_weights, grad_biases):
        for layer in range(len(self.weights)):
            self.weights[layer] -= self.learning_rate * grad_weights[layer]
            self.biases[layer] -= self.learning_rate * grad_biases[layer]

    def fit(self, X, y):
        # Gradiente full-batch: uma atualização por época, sem embaralhamento.
        logits, activations = self.forward(X)
        self.loss_history = [self.loss(logits, y)]
        for _ in range(self.epochs):
            gradients = self.backward(activations, y)
            self.update(*gradients)
            logits, activations = self.forward(X)
            self.loss_history.append(self.loss(logits, y))
        return self

    def predict(self, X):
        probabilities = self.forward(X)[1][-1]
        if self.layer_sizes[-1] == 1:
            return (probabilities[:, 0] >= 0.5).astype(int)
        return probabilities.argmax(axis=1)

    @property
    def n_parameters(self):
        return sum(W.size + b.size for W, b in zip(self.weights, self.biases))

In [5]:
binary = MLP([2, 16, 1], rng, learning_rate=LEARNING_RATE, epochs=EPOCHS)
binary.fit(Xb_train, yb_train)
results["binary_accuracy"] = accuracy(binary, Xb_test, yb_test)
results["binary_loss"] = binary.loss_history[-1]
print(f"MLP binária: {100 * results['binary_accuracy']:.2f}%")
print(f"Parâmetros: {binary.n_parameters}; loss final: {binary.loss_history[-1]:.6f}")
loss_plot(binary.loss_history, "Figura 2 — Treinamento da MLP binária",
          "MLP 2–16–1 (classes 0 e 1)", "fig02-loss-binaria.png")

# A malha está nas coordenadas originais; a rede recebe a mesma padronização do treino.
x1 = np.linspace(X_binary[:, 0].min() - 0.5, X_binary[:, 0].max() + 0.5, 300)
x2 = np.linspace(X_binary[:, 1].min() - 0.5, X_binary[:, 1].max() + 0.5, 300)
grid_x1, grid_x2 = np.meshgrid(x1, x2)
grid = np.column_stack([grid_x1.ravel(), grid_x2.ravel()])
grid_scaled = (grid - mean_binary) / std_binary
regions = binary.predict(grid_scaled).reshape(grid_x1.shape)
probabilities = binary.forward(grid_scaled)[1][-1][:, 0].reshape(grid_x1.shape)
fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(grid_x1, grid_x2, regions, levels=[-0.5, 0.5, 1.5],
            cmap=ListedColormap(COLORS[:2]), alpha=0.18)
ax.contour(grid_x1, grid_x2, probabilities, levels=[0.5], colors="black", linewidths=1)
for label in range(2):
    points = Xb_test_raw[yb_test == label]
    ax.scatter(points[:, 0], points[:, 1], s=24, c=COLORS[label],
               edgecolors="white", linewidths=0.3, label=f"Classe {label} (teste)")
ax.set(title="Figura 3 — Fronteira da MLP no conjunto de teste", xlabel="x1", ylabel="x2")
ax.legend()
save_figure(fig, "fig03-fronteira.png")

MLP binária: 92.00%
Parâmetros: 65; loss final: 0.113919


A rede acertou 184 dos 200 pontos, com acurácia de teste de 92,00%. A perda final de treino foi 0,113919. A figura 2 mostra a queda da perda e a figura 3 mostra a fronteira, com os pontos de teste nas coordenadas originais. A época 0 indica a perda antes da primeira atualização.

![Figura 2 — Perda de treino da MLP binária por época.](figures/fig02-loss-binaria.png)

Figura 2 — Entropia cruzada binária durante as 2.000 atualizações.

![Figura 3 — Regiões previstas pela MLP e pontos de teste das duas classes.](figures/fig03-fronteira.png)

Figura 3 — O fundo indica a classe prevista e a linha preta marca p = 0,5. As cores dos pontos indicam as classes verdadeiras.

### D — Analysis

A baseline ficou em 47,50% porque tenta resolver a separação com uma única reta. A classe 1 está em dois lados opostos da classe 0. Uma reta que isola bem um desses grupos deixa o outro junto com a classe central. Como os dois grupos da classe 1 são aproximadamente simétricos, uma direção linear também tem pouca informação para distinguir as classes pela média.

Na figura 3, a classe 0 fica em uma faixa central aproximadamente diagonal, e a classe 1 ocupa regiões nos dois lados dessa faixa, perto dos grupos em [3; 3] e [−3; −3]. Uma reta só divide o plano em dois semiplanos, por isso não consegue deixar a classe 1 nas duas extremidades e a classe 0 entre elas. A combinação dos neurônios ocultos permite formar essas duas transições.

Ficar abaixo de 50% neste teste não significa que qualquer classificador linear seja sempre pior que o chute. O treino tem pequenas assimetrias amostrais, e a direção escolhida pela regressão logística não generalizou bem neste split. O ganho de 44,50 pontos percentuais da MLP vem da capacidade de representar a geometria. Os 16 erros restantes também combinam com a sobreposição das caudas dos grupos.

## Exercise 3

Classificação multiclasse com a mesma implementação.

### A — Generate the data

Usei make_classification com os parâmetros pedidos: 1.500 amostras, quatro features informativas, três classes e dois clusters por classe. Não acrescentei features redundantes ou repetidas. Mantive class_sep=1,0 e random_state=42.

O split estratificado deixou 1.200 amostras no treino e 300 no teste. As contagens do treino são 399, 399 e 402 para as classes 0, 1 e 2; no teste são 100 por classe. A pequena diferença no treino ocorre porque o gerador mantém seu ruído de rótulo padrão. Novamente, a padronização foi ajustada apenas no treino.

In [6]:
X_multi, y_multi = make_classification(
    n_samples=1500, n_features=4, n_informative=4, n_redundant=0,
    n_repeated=0, n_classes=3, n_clusters_per_class=2, class_sep=1.0,
    random_state=42)
Xm_train_raw, Xm_test_raw, ym_train, ym_test = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42, stratify=y_multi)
Xm_train, Xm_test, mean_multi, std_multi = standardize(Xm_train_raw, Xm_test_raw)
print("Multiclasse: treino", Xm_train.shape, "; teste", Xm_test.shape)
print("Classes no treino/teste:", np.bincount(ym_train), np.bincount(ym_test))

Multiclasse: treino (1200, 4) ; teste (300, 4)
Classes no treino/teste: [399 399 402] [100 100 100]


### B — Adapt the network

A arquitetura agora é 4–32–3. As camadas ocultas continuam com tanh, mas a saída tem três logits, transformados em probabilidades pela softmax. A perda passa a ser a entropia cruzada categórica. Essa escolha é feita pelo número de saídas dentro da mesma classe MLP, sem alterar os métodos entre os exercícios.

Para uma amostra com alvo one-hot t, a probabilidade da classe k é pₖ = exp(uₖ)/∑ⱼexp(uⱼ). Como ∑ₖtₖ = 1, posso escrever a perda como:

$$
\ell = -\sum_k t_k\log p_k
= -\sum_k t_k u_k + \log\left(\sum_j e^{u_j}\right).
$$

Derivando em relação ao logit uᵣ:

$$
\frac{\partial\ell}{\partial u_r}
= -t_r + \frac{e^{u_r}}{\sum_j e^{u_j}} = p_r-t_r.
$$

Para a média das N amostras, ∂L/∂U = (P−T)/N. É a mesma forma do gradiente da saída binária. Por isso o restante do backward e o update não precisam mudar. No código subtraio o maior logit de cada linha antes das exponenciais, o que mantém a softmax igual e evita overflow.

### C — Train and evaluate

Treinei a rede 4–32–3 com taxa 0,1, 2.000 épocas, gradiente full-batch, inicialização Xavier e biases zero. Usei seed 42 para a inicialização. São 259 parâmetros: 4×32 + 32 + 32×3 + 3. Mantive tanh, padronização, ausência de regularização e o mesmo otimizador do exercício anterior.

In [7]:
# Reutilizamos o mesmo rng. Reiniciar seu estado permite controlar a comparação
# de profundidade; nenhum peso depende do teste ou de uma busca de hiperparâmetros.
rng.bit_generator.state = np.random.PCG64(42).state
shallow = MLP([4, 32, 3], rng, learning_rate=LEARNING_RATE, epochs=EPOCHS)
shallow.fit(Xm_train, ym_train)
results["shallow_accuracy"] = accuracy(shallow, Xm_test, ym_test)
results["shallow_loss"] = shallow.loss_history[-1]
matrix = confusion_matrix(ym_test, shallow.predict(Xm_test), labels=[0, 1, 2])
results["confusion_matrix"] = matrix.tolist()
print(f"MLP multiclasse: {100 * results['shallow_accuracy']:.2f}%")
print(f"Parâmetros: {shallow.n_parameters}; loss final: {shallow.loss_history[-1]:.6f}")
print("Matriz de confusão:\n", matrix)
loss_plot(shallow.loss_history, "Figura 4 — Treinamento da MLP multiclasse",
          "MLP 4–32–3 (classes 0, 1 e 2)", "fig04-loss-multiclasse.png")

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(matrix, cmap="Blues", vmin=0)
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(matrix[i, j]), ha="center", va="center",
                color="white" if matrix[i, j] > matrix.max() / 2 else "black")
labels = [f"Classe {i}" for i in range(3)]
ax.set(xticks=range(3), yticks=range(3), xticklabels=labels, yticklabels=labels,
       xlabel="Classe prevista", ylabel="Classe verdadeira", title="Figura 5 — Matriz de confusão (teste)")
# A escala de cor indica contagens; a legenda identifica as classes nos eixos.
ax.legend(handles=[Patch(facecolor="none", edgecolor="none", label=label) for label in labels],
          loc="upper left", bbox_to_anchor=(1.22, 1), title="Classes")
fig.colorbar(im, ax=ax, label="Número de amostras", fraction=0.046, pad=0.04)
save_figure(fig, "fig05-confusao.png")

MLP multiclasse: 84.67%
Parâmetros: 259; loss final: 0.344088
Matriz de confusão:
 [[86  6  8]
 [11 81  8]
 [ 5  8 87]]


A acurácia no teste foi 84,67%, com 254 acertos entre 300 amostras, e a perda final de treino foi 0,344088. Na figura 5, as linhas são classes verdadeiras e as colunas são classes previstas. A diagonal contém 86 acertos da classe 0, 81 da classe 1 e 87 da classe 2.

![Figura 4 — Perda de treino da MLP multiclasse por época.](figures/fig04-loss-multiclasse.png)

Figura 4 — Entropia cruzada categórica da rede com uma camada oculta.

![Figura 5 — Matriz de confusão das três classes no teste.](figures/fig05-confusao.png)

Figura 5 — Contagens das previsões, com 100 amostras verdadeiras de cada classe.

### D — Analysis

O par mais confundido é 0 e 1: seis amostras da classe 0 foram previstas como 1 e 11 da classe 1 foram previstas como 0, somando 17 erros. O par 1 e 2 soma 16 erros e o par 0 e 2 soma 13. Portanto, a diferença entre os dois pares mais confundidos é pequena, de apenas uma amostra.

Os dados têm quatro dimensões informativas e seis clusters, sendo dois por classe. Cada classe pode ocupar mais de uma região do espaço, e alguns clusters de classes diferentes podem ficar próximos. Isso ajuda a explicar a confusão entre 0 e 1. A matriz sozinha não permite apontar quais clusters se sobrepõem, então essa é uma explicação pela estrutura do gerador, e não uma localização medida dos erros. O ruído de rótulo padrão também limita a separação.

A regressão logística no mesmo split teve 66,33% de acurácia, ou 199 acertos. A MLP ganhou 18,33 pontos percentuais. A camada oculta constrói combinações não lineares das quatro entradas, permitindo fronteiras mais flexíveis para as regiões de uma mesma classe. Uma separação linear multiclasse não representa essa organização com a mesma flexibilidade.

In [8]:
baseline_multi = LogisticRegression(random_state=42).fit(Xm_train, ym_train)
results["baseline_multi"] = accuracy(baseline_multi, Xm_test, ym_test)
pairs = {(i, j): int(matrix[i, j] + matrix[j, i])
         for i in range(3) for j in range(i + 1, 3)}
results["most_confused_pair"] = list(max(pairs, key=pairs.get))
print(f"Baseline linear multiclasse: {100 * results['baseline_multi']:.2f}%")
print("Erros somados por par:", pairs)

Baseline linear multiclasse: 66.33%
Erros somados por par: {(0, 1): 17, (0, 2): 13, (1, 2): 16}


## Exercise 4

Comparação com uma segunda camada oculta.

### A — Train the deeper network

Acrescentei uma camada oculta de 32 neurônios, ficando com a arquitetura 4–32–32–3. Mantive exatamente o dataset, o split, a padronização, a taxa 0,1, as 2.000 épocas, a tanh, a inicialização Xavier e o gradiente full-batch. Reiniciei o mesmo rng na seed 42 antes de cada arquitetura. Assim, os pesos da primeira camada começam iguais; a camada adicional e a saída têm dimensões diferentes.

In [9]:
# Mesmo estado inicial do exercício 3, dados, taxa, épocas e otimizador.
# As primeiras camadas têm pesos idênticos; a nova camada acrescenta parâmetros.
rng.bit_generator.state = np.random.PCG64(42).state
deep = MLP([4, 32, 32, 3], rng, learning_rate=LEARNING_RATE, epochs=EPOCHS)
deep.fit(Xm_train, ym_train)
results["deep_accuracy"] = accuracy(deep, Xm_test, ym_test)
results["deep_loss"] = deep.loss_history[-1]
print(f"MLP com duas camadas ocultas: {100 * results['deep_accuracy']:.2f}%")
print(f"Parâmetros: {deep.n_parameters}; loss final: {deep.loss_history[-1]:.6f}")

MLP com duas camadas ocultas: 86.67%
Parâmetros: 1315; loss final: 0.276304


A rede mais profunda acertou 260 de 300 pontos, com acurácia de teste de 86,67%. A perda final de treino foi 0,276304. Ela tem 1.315 parâmetros, calculados por (4×32 + 32) + (32×32 + 32) + (32×3 + 3).

### B — Compare

A figura 6 compara as perdas das duas redes na seed 42. As duas curvas usam a mesma quantidade de atualizações. A segunda camada reduz mais a perda neste treinamento.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(shallow.loss_history, label="1 camada oculta (classes 0, 1 e 2)")
ax.plot(deep.loss_history, label="2 camadas ocultas (classes 0, 1 e 2)")
ax.set(title="Figura 6 — Comparação de profundidade (seed 42)",
       xlabel="Época (0 = antes de treinar)", ylabel="Entropia cruzada média")
ax.legend()
save_figure(fig, "fig06-comparacao.png")

seed_rows = []
for seed in [42, 43, 44]:
    for architecture in [[4, 32, 3], [4, 32, 32, 3]]:
        if seed == 42:
            # Estes dois modelos já foram treinados acima.
            model = shallow if len(architecture) == 3 else deep
        else:
            # Só a inicialização varia: dataset e split continuam em random_state=42.
            # Mantemos um único Generator; mudamos seu estado para cada seed pareada.
            rng.bit_generator.state = np.random.PCG64(seed).state
            model = MLP(architecture, rng, learning_rate=LEARNING_RATE, epochs=EPOCHS)
            model.fit(Xm_train, ym_train)
        seed_rows.append({"seed": seed, "hidden_layers": len(architecture) - 2,
                          "parameters": model.n_parameters,
                          "train_loss": model.loss_history[-1],
                          "test_accuracy": accuracy(model, Xm_test, ym_test)})
seed_results = pd.DataFrame(seed_rows)
means = seed_results.groupby("hidden_layers")[["train_loss", "test_accuracy"]].mean()
results["shallow_mean_accuracy"] = float(means.loc[1, "test_accuracy"])
results["deep_mean_accuracy"] = float(means.loc[2, "test_accuracy"])
results["mean_accuracy_difference"] = results["deep_mean_accuracy"] - results["shallow_mean_accuracy"]
seed_results.to_csv(BASE / "code" / "seed-results.csv", index=False)
pd.DataFrame({"epoch": np.arange(EPOCHS + 1), "binary": binary.loss_history,
              "shallow": shallow.loss_history, "deep": deep.loss_history}).to_csv(
                  BASE / "code" / "loss-history.csv", index=False)
print(seed_results.to_string(index=False))
print("Médias:\n", means)
print(f"Diferença média (profunda - rasa): {100 * results['mean_accuracy_difference']:.2f} p.p.")

 seed  hidden_layers  parameters  train_loss  test_accuracy
   42              1         259    0.344088       0.846667
   42              2        1315    0.276304       0.866667
   43              1         259    0.356575       0.840000
   43              2        1315    0.270152       0.873333
   44              1         259    0.340917       0.846667
   44              2        1315    0.277255       0.856667
Médias:
                train_loss  test_accuracy
hidden_layers                           
1                0.347194       0.844444
2                0.274570       0.865556
Diferença média (profunda - rasa): 2.11 p.p.


: 

![Figura 6 — Perdas das redes com uma e duas camadas ocultas.](figures/fig06-comparacao.png)

Figura 6 — Comparação com os mesmos dados, seed 42, taxa de aprendizado e número de épocas.

A comparação da execução com seed 42 é:

| Modelo | Camadas ocultas | Parâmetros | Perda final de treino | Acurácia de teste |
|---|---|---|---|---|
| Exercise 3 | 1 | 259 | 0,344088 | 84,67% |
| Exercise 4 | 2 | 1.315 | 0,276304 | 86,67% |

Para verificar o quanto isso dependia da inicialização, repeti as duas arquiteturas com seeds 42, 43 e 44. Só mudei a seed dos pesos. Os dados e o split continuaram com random_state=42, e não fiz busca de hiperparâmetros no teste.

| Seed | Perda final: 1 camada | Perda final: 2 camadas | Acurácia: 1 camada | Acurácia: 2 camadas | Diferença |
|---|---|---|---|---|---|
| 42 | 0,344088 | 0,276304 | 84,67% | 86,67% | +2,00 p.p. |
| 43 | 0,356575 | 0,270152 | 84,00% | 87,33% | +3,33 p.p. |
| 44 | 0,340917 | 0,277255 | 84,67% | 85,67% | +1,00 p.p. |
| Média | 0,347194 | 0,274570 | 84,44% | 86,56% | +2,11 p.p. |

As médias foram calculadas com os números completos, antes de arredondar. A rede profunda ficou melhor nas três inicializações, com ganho médio de 2,11 pontos percentuais. O arquivo code/seed-results.csv guarda os resultados individuais e code/loss-history.csv guarda as curvas.

### C — Analysis

Neste experimento a segunda camada ajudou um pouco, mesmo que o enunciado indique que ela provavelmente não ajudaria. Mantive o resultado medido: 86,56% contra 84,44% na média das três seeds. A perda média também caiu, de 0,347194 para 0,274570, então a melhoria no ajuste do treino veio acompanhada de um pequeno ganho no teste.

Os seis clusters em quatro dimensões não exigem necessariamente uma rede profunda. A rede com uma camada já chegou a 84,67% na seed principal. Uma segunda camada pode recombinar os padrões que a primeira detecta e tornar a fronteira mais flexível, mas também aumenta a quantidade de parâmetros e pode dificultar a otimização, por propagar o gradiente por mais uma tanh. Aqui ela conseguiu reduzir a perda com o orçamento fixado.

O custo também cresceu: foram 1.056 parâmetros a mais, chegando a aproximadamente 5,08 vezes a quantidade da rede rasa. O número de épocas foi igual, mas cada época da rede profunda envolve mais operações. Esta comparação mede o efeito de acrescentar a camada com essa largura, incluindo os parâmetros extras; ela não isola profundidade de quantidade de parâmetros. Três seeds em um único split também não bastam para afirmar que essa arquitetura será sempre melhor.

A profundidade tende a valer mais quando os dados têm padrões que podem ser construídos em etapas, como contornos que viram partes de objetos em imagens, ou várias regiões locais cuja combinação define a classe. Neste dataset pequeno, o ganho foi modesto em relação ao aumento de capacidade. A rede rasa continua sendo uma escolha razoável se o custo de treinamento for relevante.

## Results summary

Os resultados de classificação também ficam em code/results.json. Os parâmetros do exercício 1 abaixo seguem a convenção de pesos por linha usada nas contas.

| # | Quantity | Value |
|---|---|---|
| 1 | Ex. 1 — z¹ e h¹ | z¹ = [0,270000; −0,180000]; h¹ = [0,263625; −0,178081] |
| 2 | Ex. 1 — u², ŷ e L | 0,385237; 0,367247; 0,400377 |
| 3 | Ex. 1 — parâmetros atualizados | W¹ = [[0,376405; −0,130562]; [0,152295; 0,419082]]; b¹ = [0,252811; −0,295410]; W² = [0,586587; −0,358490]; b² = 0,528448 (nove valores escalares) |
| 4 | Ex. 2 — acurácia da baseline linear | 47,50% |
| 5 | Ex. 2 — arquitetura e acurácia | 2–16–1; 92,00% |
| 6 | Ex. 3 — arquitetura e acurácia | 4–32–3; 84,67% (seed 42) |
| 7 | Ex. 3 — par de classes mais confundido | Classes 0 e 1; 17 erros entre elas |
| 8 | Ex. 4 — arquitetura e acurácia | 4–32–32–3; 86,67% (seed 42); média de 86,56% |
| 9 | Ex. 4 — diferença média de acurácia frente ao Ex. 3 | +2,11 p.p.; seeds 42, 43 e 44 (86,56% contra 84,44%) |